# 003 · Context Managers

> **Syntax › Error Handling** · Python 3.14

The `with` statement is Python's `using`: it guarantees that cleanup code runs when a block exits — normally or via an exception. Objects that work with `with` are called **context managers**, and unlike `IDisposable` they can also run code on *entry* and even decide to swallow exceptions.

**In this notebook you will learn:**

- `with` vs C#'s `using` / `IDisposable`
- reading and writing files safely with `open()`
- writing a class-based context manager with `__enter__` / `__exit__`, including exception suppression
- the simpler generator form with `@contextlib.contextmanager`
- using several managers in one `with` statement
- `contextlib.suppress` and `contextlib.ExitStack`
- a pointer to `async with`

## C# mental model

| C# | Python |
|---|---|
| `IDisposable` | the context manager protocol: `__enter__` / `__exit__` |
| `using (var x = new Foo()) { }` | `with Foo() as x:` |
| `using var x = new Foo();` (declaration) | no equivalent — `with` always has a block |
| `Dispose()` | `__exit__(exc_type, exc, tb)` |
| constructor / "acquire" | `__enter__()` — its return value is bound by `as` |
| `IAsyncDisposable` + `await using` | `__aenter__` / `__aexit__` + `async with` |
| `try { } catch (X) { }` (ignore) | `with contextlib.suppress(X):` |

## `with` and files

```csharp
// C#
using (var writer = new StreamWriter(path))
{
    writer.WriteLine("hello");
}   // writer.Dispose() is called here, even if an exception was thrown
```

`open()` returns a file object that is also a context manager: the file is closed when the `with` block exits. We'll work in a temporary directory so nothing is written into the project.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:          # TemporaryDirectory is a context manager too
    path = Path(tmp) / "notes.txt"

    with open(path, "w", encoding="utf-8") as f:
        f.write("first line\n")
        f.write("second line\n")
    print("closed after with:", f.closed)

    with open(path, encoding="utf-8") as f:         # default mode is "r" (read text)
        for line in f:                              # files iterate line by line
            print(line.rstrip())

print("temp dir removed:", not Path(tmp).exists())

Without `with`, you'd need `try`/`finally` to be safe — exactly what `using` saves you from writing in C#:

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "data.txt"
    path.write_text("42\n", encoding="utf-8")

    f = open(path, encoding="utf-8")
    try:
        print(f.read().strip())
    finally:
        f.close()                                   # what 'with' does for you
    print("closed:", f.closed)

⚠️ **Gotcha:** the `as` variable **outlives** the `with` block (Python has no block scope), but the resource behind it has been released. Using it afterwards raises an error — similar to `ObjectDisposedException`.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "data.txt"
    path.write_text("hello", encoding="utf-8")
    with open(path, encoding="utf-8") as f:
        pass

    try:
        f.read()
    except ValueError as e:
        print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** always pass `encoding="utf-8"` when opening text files. The default encoding is platform-dependent (it may not be UTF-8 on Windows), unlike .NET, which defaults to UTF-8.

## Writing a class-based context manager

Any class with `__enter__` and `__exit__` methods works with `with`:

- `__enter__(self)` runs when the block starts. Its **return value** is what `as` binds (often `self`, but not necessarily).
- `__exit__(self, exc_type, exc, tb)` runs when the block ends. If the block raised, the three arguments describe the exception; otherwise they're all `None`.

```csharp
// C#
sealed class Timer : IDisposable
{
    private readonly Stopwatch _sw = Stopwatch.StartNew();
    public void Dispose() => Console.WriteLine($"took {_sw.ElapsedMilliseconds} ms");
}
```

In [ ]:
import time

class Timer:
    def __init__(self, label: str) -> None:
        self.label = label

    def __enter__(self) -> "Timer":
        print(f"[{self.label}] start")
        self.start = time.perf_counter()
        return self                                 # bound to the 'as' target

    def __exit__(self, exc_type, exc, tb) -> None:
        self.elapsed = time.perf_counter() - self.start
        status = "failed" if exc_type else "ok"
        print(f"[{self.label}] {status}")

with Timer("sum") as t:
    total = sum(range(1_000_000))
print("total:", total, "| elapsed < 1s:", t.elapsed < 1)

`__exit__` runs even when the block raises — and the exception continues to propagate afterwards:

In [ ]:
try:
    with Timer("division"):
        1 / 0
except ZeroDivisionError as e:
    print(f"caught outside: {type(e).__name__}: {e}")

### Suppressing exceptions: `return True`

Here's something `IDisposable` can't do: if `__exit__` returns a **truthy** value, the exception is **swallowed** and execution continues after the `with` block. Returning `None`/`False` lets it propagate.

In [ ]:
class IgnoreErrors:
    def __init__(self, *types: type[BaseException]) -> None:
        self.types = types

    def __enter__(self) -> None:
        return None

    def __exit__(self, exc_type, exc, tb) -> bool:
        if exc_type is not None and issubclass(exc_type, self.types):
            print(f"  suppressed {exc_type.__name__}: {exc}")
            return True                             # swallow it
        return False                                # propagate everything else

with IgnoreErrors(KeyError):
    {}["missing"]
    print("never printed — the block stops at the exception")
print("execution continues after the with block")

try:
    with IgnoreErrors(KeyError):
        int("abc")                                  # ValueError is not suppressed
except ValueError as e:
    print(f"propagated: {type(e).__name__}: {e}")

⚠️ **Gotcha:** suppression is powerful but easy to misuse. An `__exit__` that accidentally returns a truthy value hides every error in the block. Only return `True` deliberately.

### A realistic example: a transaction

A context manager that commits on success and rolls back on failure — the same pattern as `TransactionScope` in .NET, where "complete" is decided by whether an exception occurred.

In [ ]:
class Transaction:
    def __init__(self, store: dict) -> None:
        self.store = store

    def __enter__(self) -> dict:
        self.snapshot = dict(self.store)            # remember the state
        return self.store

    def __exit__(self, exc_type, exc, tb) -> None:
        if exc_type is None:
            print("  commit")
        else:
            self.store.clear()
            self.store.update(self.snapshot)        # roll back
            print(f"  rollback ({exc_type.__name__})")

accounts = {"alice": 100, "bob": 50}

with Transaction(accounts) as acc:
    acc["alice"] -= 30
    acc["bob"] += 30
print(accounts)

try:
    with Transaction(accounts) as acc:
        acc["alice"] -= 500
        if acc["alice"] < 0:
            raise ValueError("insufficient funds")
except ValueError:
    pass
print(accounts)                                     # unchanged by the failed transfer

## The generator form: `@contextlib.contextmanager`

Writing a class is verbose. The `@contextmanager` decorator turns a **generator function** *(see Iterators & Generators › 002 · Generators)* into a context manager:

- code **before** `yield` is the `__enter__` part,
- the **yielded value** is bound by `as`,
- code **after** `yield` is the `__exit__` part.

Wrap the `yield` in `try`/`finally` so cleanup runs even if the block raises — the exception is re-raised *at the `yield`*.

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer(label: str):
    print(f"[{label}] start")
    start = time.perf_counter()
    try:
        yield start                                 # the 'as' value
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{label}] done ({'fast' if elapsed < 1 else 'slow'})")

with timer("loop") as started_at:
    print("started_at is a float:", isinstance(started_at, float))
    sum(range(100_000))

In [ ]:
@contextmanager
def tag(name: str):
    print(f"<{name}>")
    try:
        yield
    finally:
        print(f"</{name}>")

with tag("ul"):
    with tag("li"):
        print("item")

To suppress an exception in the generator form, catch it around the `yield` and don't re-raise:

In [ ]:
@contextmanager
def ignore(*types):
    try:
        yield
    except types as e:
        print(f"  ignored {type(e).__name__}")

with ignore(ZeroDivisionError):
    1 / 0
print("still running")

⚠️ **Gotcha:** without `try`/`finally`, the code after `yield` is **skipped** when the block raises — a classic resource leak:

In [ ]:
@contextmanager
def leaky():
    print("acquire")
    yield
    print("release")                                # skipped on exception!

try:
    with leaky():
        raise RuntimeError("boom")
except RuntimeError as e:
    print(f"{type(e).__name__}: {e} — and 'release' was never printed")

## Multiple managers in one `with`

Separate managers with commas. Since 3.10 you can wrap them in parentheses to span several lines. They're entered left to right and exited in **reverse** order — like stacked `using` statements in C#.

```csharp
// C#
using var source = File.OpenRead(src);
using var target = File.Create(dst);
source.CopyTo(target);
```

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    src, dst = Path(tmp) / "in.txt", Path(tmp) / "out.txt"
    src.write_text("alice\nbob\ncarol\n", encoding="utf-8")

    with (
        open(src, encoding="utf-8") as source,
        open(dst, "w", encoding="utf-8") as target,
    ):
        for line in source:
            target.write(line.upper())

    print(dst.read_text(encoding="utf-8"))

In [ ]:
with tag("outer"), tag("inner"):                   # exit order is reversed
    print("body")

## `contextlib.suppress`

`suppress(*exceptions)` is the built-in version of our `IgnoreErrors` class — a concise way to say "this error is expected and fine". It replaces the C# pattern of an empty `catch`.

```csharp
// C#
try { File.Delete(path); } catch (FileNotFoundException) { }
```

In [ ]:
from contextlib import suppress
import os

with suppress(FileNotFoundError):
    os.remove("/no/such/file.tmp")
print("no error")

with suppress(KeyError, IndexError):
    [][0]
print("still no error")

## `contextlib.ExitStack` (briefly)

When the **number** of resources is only known at runtime, you can't write them all in one `with`. `ExitStack` lets you register any number of context managers (or plain callbacks) and unwinds them all, in reverse order, when the block exits.

In [ ]:
from contextlib import ExitStack

with tempfile.TemporaryDirectory() as tmp:
    names = ["a.txt", "b.txt", "c.txt"]
    for name in names:
        (Path(tmp) / name).write_text(f"content of {name}", encoding="utf-8")

    with ExitStack() as stack:
        files = [stack.enter_context(open(Path(tmp) / n, encoding="utf-8")) for n in names]
        stack.callback(print, "callback: all files about to be closed")
        print([f.read() for f in files])

    print("all closed:", all(f.closed for f in files))

## Other handy context managers

The standard library has many ready-made context managers. A few common ones:

| Context manager | Purpose |
|---|---|
| `open(...)` | files |
| `tempfile.TemporaryDirectory()` / `NamedTemporaryFile()` | temporary files and folders |
| `threading.Lock()` | `with lock:` — like C#'s `lock (obj) { }` |
| `contextlib.suppress(...)` | ignore specific exceptions |
| `contextlib.closing(obj)` | call `obj.close()` on exit |
| `contextlib.chdir(path)` (3.11+) | temporarily change the working directory |
| `warnings.catch_warnings()` | temporarily change warning filters |

In [ ]:
import threading

lock = threading.Lock()
with lock:                                          # C#: lock (syncRoot) { ... }
    print("inside critical section, locked:", lock.locked())
print("after:", lock.locked())

## `async with` (a pointer)

Async resources (network connections, async locks, task groups) use `async with`, which calls `__aenter__` / `__aexit__` — the counterpart of C#'s `await using` with `IAsyncDisposable`. There's also `@contextlib.asynccontextmanager`. Details in *Async › 001 · Async and Await* and *Async › 002 · Running Tasks Concurrently*.

```csharp
// C#
await using var stream = File.OpenRead(path);   // IAsyncDisposable
```

```python
# Python
async with asyncio.TaskGroup() as tg:     # waits for all tasks on exit
    tg.create_task(fetch("a"))
    tg.create_task(fetch("b"))
```

## Summary

| Task | C# | Python |
|---|---|---|
| Scoped cleanup | `using (var x = ...) { }` | `with ... as x:` |
| Implement | `IDisposable.Dispose()` | `__enter__` + `__exit__` |
| Quick implementation | — | `@contextmanager` generator |
| Swallow an exception on exit | not possible with `Dispose` | `__exit__` returns `True` |
| Several resources | stacked `using` | `with (a as x, b as y):` |
| Ignore an exception | empty `catch` | `with suppress(X):` |
| Dynamic number of resources | `List<IDisposable>` + loop | `ExitStack` |
| Lock | `lock (obj) { }` | `with lock:` |
| Async | `await using` | `async with` |

## Exercises

1. Write a `@contextmanager` called `indented(prefix)` that sets a module-level `PREFIX` variable for the duration of the block and restores the previous value afterwards — even if the block raises.
2. Write a class-based context manager `Counter` whose `__exit__` counts how many times the block raised an exception (`self.failures`) and suppresses only `ValueError`.
3. In a `TemporaryDirectory`, write three lines to a file, then read it back and count the lines — using `with` for every resource.
4. What happens if `__exit__` accidentally returns a non-empty string?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
from contextlib import contextmanager

PREFIX = ""

@contextmanager
def indented(prefix: str):
    global PREFIX
    previous = PREFIX
    PREFIX = prefix
    try:
        yield
    finally:
        PREFIX = previous

# 2
class Counter:
    def __init__(self) -> None:
        self.failures = 0

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb) -> bool:
        if exc_type is not None:
            self.failures += 1
        return exc_type is not None and issubclass(exc_type, ValueError)

# 3
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "lines.txt"
    with open(path, "w", encoding="utf-8") as f:
        f.write("one\ntwo\nthree\n")
    with open(path, encoding="utf-8") as f:
        print(sum(1 for _ in f))    # 3

# 4 — a non-empty string is truthy, so any exception raised in the block
#     would be silently suppressed.
```

</details>